<a href="https://colab.research.google.com/github/cyeef/Capstone-Projects/blob/main/Cloud-App/cloud_based_diagnostic_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

## Load the NLM SNOMED to ICD-10-CM Map

### Step 1: Mapping SNOMED CT Codes to ICD-10-CM
To evaluate whether this proposed rural triage unit can maintain financial solvency, we must first translate raw clinical condition records into billable diagnostic codes. This step ingests the NLM SNOMED CT to ICD-10-CM map ([NLM UMLS Map](https://www.nlm.nih.gov/research/umls/mapping_projects/snomedct_to_icd10cm.html), accessed August 2026) and evaluates its conditional rules (such as age and sex criteria) to convert patient condition codes. This is performed strictly as a sustainability demonstration to establish encounter volume, rather than a clinical revenue optimizer.

In [2]:
# Load the NLM SNOMED to ICD-10-CM Map
try:
    # Updated to the exact file extracted from your downloaded folder
    nlm_map = pd.read_csv('der2_iisssccRefset_ExtendedMapSnapshot_US1000124_20260301.txt', sep='\t', dtype=str)
except FileNotFoundError:
    print("Warning: NLM map file not found. Please upload it to Colab.")
    nlm_map = pd.DataFrame(columns=['referencedComponentId', 'mapGroup', 'mapPriority', 'mapRule', 'mapTarget'])

def map_snomed_to_icd10(snomed_code, age, sex, map_df):
    """
    Applies NLM rules to map a SNOMED code to an ICD-10-CM code.
    """
    # Filter for the specific SNOMED concept
    concept_rules = map_df[map_df['referencedComponentId'] == str(snomed_code)]

    if concept_rules.empty:
        return None, "Unmapped: SNOMED code not in NLM map"

    # Sort by group and priority to evaluate rules in the correct order
    concept_rules = concept_rules.sort_values(by=['mapGroup', 'mapPriority'])

    # Evaluate rules
    for _, row in concept_rules.iterrows():
        rule = row['mapRule']
        target = row['mapTarget']

        # If no target exists for this rule path
        if pd.isna(target) or target.strip() == "":
            continue

        # Clean rule evaluation
        if rule == "OTHERWISE TRUE" or rule == "TRUE":
            return target, "Mapped cleanly/fallback"

        # Conditional rule evaluation (IFA)
        if "IFA" in rule:
            # Example rule: 'IFA 248152002 | Female |'
            if "Female" in rule and sex.lower() == 'f':
                return target, "Mapped with sex rule"
            elif "Male" in rule and sex.lower() == 'm':
                return target, "Mapped with sex rule"

    return None, "Unmapped: No valid rule target met"

## ICD-10-CM Validation

### Step 2: Validating ICD-10-CM Codes Against Official Standards
Before any mapped diagnostic codes are passed further down the reimbursement chain, they must be verified to prevent invalid values from propagating silently. This block parses the official fixed-width text files from the [CDC/NCHS ICD-10-CM 2027 Set](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10CM/2027/) (accessed August 2026) to confirm that every generated code actively exists in the official federal dataset. Unmapped or unrecognized codes are explicitly tracked rather than silently dropped to ensure data transparency.

In [3]:
# Parse the CDC Fixed-Width File
# Adjust the colspecs based on inspection of the actual 2027 txt file.
# ICD-10 codes are in the first 7 characters, followed by a space, then the description.
try:
    col_positions = [(0, 7), (8, 200)]
    cdc_icd10 = pd.read_fwf('/content/icd10cm_order_2027.txt', colspecs=col_positions, header=None, names=['code', 'description'])
    valid_icd10_codes = set(cdc_icd10['code'].str.strip())
except FileNotFoundError:
    print("Warning: CDC ICD-10 file not found.")
    valid_icd10_codes = set()

def validate_icd10(icd10_code):
    """Checks if the ICD-10 code exists in the official CDC set."""
    if icd10_code is None:
        return False
    # Strip decimals if the NLM map includes them, as CDC files usually drop the decimal
    clean_code = str(icd10_code).replace('.', '').strip()
    return clean_code in valid_icd10_codes

## DRG Approximation & Payment Weight Estimate

### Step 3 & 4: Approximating DRGs and Estimating Payment Weight
*Methodological Limitation:* Inpatient Diagnosis-Related Groups (DRGs) are officially assigned per complete hospital stay—considering principal diagnoses, secondary complications, and procedures—rather than a single point-of-care diagnosis. Because this tool acts as a rapid rural triage approximation, we use the patient's primary condition as a practical diagnostic proxy matched via the [NBER DRG–MDC Crosswalk](https://www.nber.org/research/data/diagnosis-related-group-major-diagnostic-category-crosswalk) (accessed August 2026).

Using the retrieved relative weight multiplied against the documented CMS Base Rate (FY2024 IPPS standardized amount of $6,497.77, sourced from [CMS.gov](https://www.cms.gov)), this code calculates an estimated encounter value to evidence financial viability.

### Loading file to inspect Columns

In [4]:
# Load the file to inspect columns
nber_crosswalk = pd.read_csv('/content/drgweight2026FR.csv')
print("Available columns in NBER file:")
display(pd.DataFrame(nber_crosswalk.columns.tolist(), columns=["Column Name"]))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drgweight2026FR.csv'

### Crosswalk Logic

Methodological Limitation: In this notebook, I approximate the DRG using the patient's primary condition as a proxy. Real DRG assignment requires a comprehensive "grouper" that considers the entire inpatient stay (procedures, comorbidities, discharge status). This tool is strictly a sustainability approximation, not an exact revenue or grouping assignment.

In [ ]:
# CMS Base Rate (Example standardized amount)
CMS_BASE_RATE = 6497.77

try:
    # 1. Loading the NBER DRG weight file using confirmed filename
    nber_df = pd.read_csv('/content/drgweight2026FR.csv')

    # 2. Standardizing ms_drg to a zero-padded 3-digit string for consistent matching
    nber_df['ms_drg'] = nber_df['ms_drg'].astype(str).str.zfill(3)

    # 3. Creating a dictionary lookup indexed by 'ms_drg'
    drg_lookup = nber_df.set_index('ms_drg').to_dict('index')
    print(f"Successfully loaded {len(drg_lookup)} DRGs from NBER data.")

except FileNotFoundError:
    print("Warning: NBER file not found. Check your file path.")
    drg_lookup = {}

def get_drg_payment_info(ms_drg_code):
    """Retrieves weight, title, and MDC based on the assigned MS-DRG code."""
    clean_drg = str(ms_drg_code).strip().str.zfill(3) if pd.notna(ms_drg_code) else None

    if clean_drg in drg_lookup:
        data = drg_lookup[clean_drg]
        weight = data.get('weights')
        title = data.get('msdrg_title')
        mdc = data.get('mdc')

        # Calculating estimated payment
        est_payment = weight * CMS_BASE_RATE if pd.notna(weight) else None
        return weight, title, mdc, est_payment

    return None, None, None, None

# Example usage on a patient dataframe already containing an assigned MS-DRG column

# patient_df['weight'], patient_df['drg_title'], patient_df['mdc'], patient_df['estimated_payment'] = zip(*patient_df['assigned_ms_drg'].apply(get_drg_payment_info))

## The Interface Function

#### Wrap everything into a strict dictionary signature

### The Final Interface

### Step 5: Constructing the Final Encounter Value Interface
To seamlessly integrate this evaluation pipeline into the existing Streamlit demonstration app without requiring major structural refactoring, all previous mapping, validation, and weighting logic is wrapped into a single Python function: `estimate_encounter_value(snomed_codes, age, sex)`. This function processes a patient cohort, tracks structural unmapped gaps, and returns a strictly formatted dictionary containing the estimated reimbursement figures alongside mandatory non-clinical caveats.

In [ ]:
import json

def format_encounter_result(result_dict):
    """
    Takes the dictionary from estimate_encounter_value and prints it
    in a clean, human-readable, structured format.
    """
    print("=" * 60)
    print("           PATIENT ENCOUNTER REIMBURSEMENT ESTIMATE          ")
    print("=" * 60)

    # Validated Codes
    print("\n[+] Mapped & Validated ICD-10 Codes:")
    if result_dict['icd10_codes']:
        for code in result_dict['icd10_codes']:
            print(f"    • {code}")
    else:
        print("    (None)")

    # Unmapped Codes
    print("\n[-] Unmapped / Failed Codes:")
    if result_dict['unmapped']:
        for item in result_dict['unmapped']:
            print(f"    • SNOMED: {item['snomed']} | Reason: {item['reason']}")
    else:
        print("    (None)")

    # Financial & Grouping Summary
    print("\n[#] Reimbursement & Grouping Summary:")
    print(f"    • Approximated DRG : {result_dict['drg'] or 'N/A'}")
    print(f"    • Major Diagnostic Category (MDC): {result_dict['mdc'] or 'N/A'}")
    print(f"    • DRG Relative Weight: {result_dict['weight'] or 'N/A'}")

    est_usd = result_dict['estimate_usd']
    formatted_usd = f"${est_usd:,.2f}" if est_usd is not None else "N/A"
    print(f"    • Estimated Value    : {formatted_usd}")

    # Mandatory Caveats
    print("\n[!] Methodological Caveats:")
    for caveat in result_dict['caveats']:
        print(f"    * {caveat}")
    print("=" * 60)

# Example usage with your dictionary output:
sample_output = {
    'icd10_codes': [],
    'unmapped': [
        {'snomed': '44054006', 'reason': 'Failed CDC Validation'},
        {'snomed': '195967001', 'reason': 'Failed CDC Validation'}
    ],
    'drg': None,
    'mdc': None,
    'weight': None,
    'estimate_usd': None,
    'caveats': [
        'NON-CLINICAL USE ONLY. Data is synthetic.',
        'DRG approximated from primary diagnosis proxy. Actual grouping requires full inpatient context.'
    ]
}

format_encounter_result(sample_output)

Checking the mapping, the decimals places for the first 20 results.

In [ ]:
print(cdc_codes[:20])            # what format does the CDC file use?
print(mapped_icd10_codes[:20])   # what format does the map produce?

# Modify the Condition extraction to keep codes alongside display names

In [ ]:
# Modify the Condition extraction to keep codes alongside display names
record['condition_codes'] = [
    c['code']['coding'][0]['code']
    for c in resources if c['resourceType'] == 'Condition'
    and c.get('code', {}).get('coding')
]

# Then in export_cohort.py, add:
app_data[case]['reimbursement'] = estimate_encounter_value(
    r['condition_codes'], r['age'], r['gender']
)

Add streamlit application to this code

In [ ]:
st.subheader("Encounter Value Estimate")
st.caption("Demonstration only — synthetic data, approximated DRG assignment.")

rb = p.get('reimbursement')
if rb:
    c1, c2, c3 = st.columns(3)
    c1.metric("DRG", rb['drg'] or "—")
    c2.metric("Relative weight", f"{rb['weight']:.4f}" if rb['weight'] else "—")
    c3.metric("Estimated value",
              f"${rb['estimate_usd']:,.0f}" if rb['estimate_usd'] else "—")

    if rb['icd10_codes']:
        st.write("**Mapped ICD-10:** " + ", ".join(rb['icd10_codes']))
    if rb['unmapped']:
        with st.expander(f"{len(rb['unmapped'])} unmapped code(s)"):
            for u in rb['unmapped']:
                st.write(f"• SNOMED {u['snomed']} — {u['reason']}")
    for c in rb['caveats']:
        st.caption(f"⚠ {c}")
else:
    st.info("No reimbursement estimate available for this patient.")